In [82]:
import pandas as pd

In [84]:
# 1. Dosyayı oku
file_path = 'tahmin-oran.csv'  # <-- kendi dosya yolunu ver
df = pd.read_csv(file_path)

In [86]:
# 2. İlgili personel türleri
personel_turleri = [
    "uzman", "asistan", "pratisyen", "diger", "hemsire", "ebe", "disci", "eczaci"
]

In [88]:
# 3. Sadece 2023 ve 2024 verilerini al
df_2023 = df[df['yıl'] == 2023]
df_2024 = df[df['yıl'] == 2024]

In [90]:
# 4. 2023-2024 arasındaki artışları şehir bazlı bul
artis_df = df_2024.copy()
for personel in personel_turleri:
    artis_df[personel] = df_2024[personel].values - df_2023[personel].values

# Şehir adını ve artış sayılarını tut
artis_df = artis_df[['il_adı'] + personel_turleri]

In [92]:
# 5. Türkiye genelinde her personel türü için toplam artışı bul (yani yeni mezun toplam sayısı gibi kullanacağız)
turkiye_artislar = artis_df[personel_turleri].sum()

In [95]:
# 6. Şehirlerin yüzdelerini hesapla
ihtiyac_yuzdeleri = artis_df.copy()

for personel in personel_turleri:
    ihtiyac_yuzdeleri[personel] = (artis_df[personel] / turkiye_artislar[personel]) * 100

In [97]:
# 7. Normalize et (her sütunun toplamı 100 olsun)
for personel in personel_turleri:
    toplam = ihtiyac_yuzdeleri[personel].sum()
    ihtiyac_yuzdeleri[personel] = ihtiyac_yuzdeleri[personel] / toplam * 100

In [99]:
# 8. Virgülden sonra 8 basamak ayarla
ihtiyac_yuzdeleri = ihtiyac_yuzdeleri.round(8)

In [101]:
# 9. Aşağı bir satır daha ekle (TOPLAMLAR)
toplam_satiri = ihtiyac_yuzdeleri[personel_turleri].sum()
toplam_satiri['il_adı'] = 'Toplam'
ihtiyac_yuzdeleri = pd.concat([ihtiyac_yuzdeleri, toplam_satiri.to_frame().T], ignore_index=True)

In [105]:
# 10. CSV'ye kaydet (float_format ile düz sayı gösterimi)
ihtiyac_yuzdeleri.to_csv(
    'ihtiyaclar.csv', 
    index=False,
    float_format='%.8f'
)

print("Her sütunun toplamı (en son satırda da var):")
print(ihtiyac_yuzdeleri.tail(2))

Her sütunun toplamı (en son satırda da var):
    il_adı       uzman     asistan   pratisyen       diger      hemsire  \
80  Şırnak  0.49792157 -0.01484637  1.10636446  0.30378480   0.04002891   
81  Toplam 99.99999999 99.99999998 99.99999998 99.99999999 100.00000001   

           ebe       disci      eczaci  
80  0.85840780  0.03438074 -0.29272823  
81 99.99999998 99.99999999 99.99999997  
